## Creating a sample of 1000 random elative construction phrases

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

import os
import csv
import random
from estnltk_core.converters.json_importer import json_to_text

from methods.helper_methods import get_subtree

In [ ]:
FILE_PATH = "../data/koondkorpus_sentences/conx_example_sentences_20042026/"

In [4]:
filenames = os.listdir(FILE_PATH)

#### I collecting 1000 random construction instances from data, assembling dataset

In [ ]:
# randomize order
random.shuffle(filenames)

In [ ]:
example_sentences = []

for idx, filename in enumerate(filenames):
    if len(example_sentences) == 1000:
        break

    sentence_text = json_to_text(file=f"{FILE_PATH}{filename}")

    # looking for elative construction phrase
    words = sentence_text.v172_stanza_syntax
    
    for stanza_word in words:
        # ignoring words that are not nominal modifiers
        if stanza_word.deprel != "nmod":
            continue
        
        morph = stanza_word.morph_analysis
        lemma = morph.lemma[0]
        pos = morph.partofspeech
        form = morph.form

        # excluding instances where nominal modifier isn't a substantive or in singular elative case
        if "S" not in pos or "sg el" not in form:
            continue
        
        # excluding instances where nominal modifier in elative case doesn't have a syntactic parent
        parent = stanza_word.parent_span
        if parent is None:
            continue
        
        # excluding instances where nominal modifier in elative case succeeds its syntactic parent
        if parent.id < stanza_word.id:
            continue
        
        # excluding instances where parent is not a substantive or proper noun
        parent_pos = parent.morph_analysis.partofspeech
        if not ("S" in parent_pos or "H" in parent_pos):
            continue
        
        # excluding instances where parent lemma ends with 'mine'
        parent_lemma = parent.morph_analysis.lemma[0]
        if parent_lemma.endswith("mine"):
            continue
        
        # excluding instances where parent is in terminative case
        parent_form = parent.morph_analysis.form
        if "sg ter" in parent_form or "pl ter" in parent_form:
            #print(parent_lemma)
            continue
        
        # finding full phrase from syntax tree
        instance_form_long_members = get_subtree(parent)

        # ordering full phrase
        instance_form_long = []
        for i in range(1, len(words)+1):
            for el in instance_form_long_members:
                if i == el.id:
                    instance_form_long.append(el.text.lower())
        
        # assembling data
        dct = {"sentence_id": int(filename.split(".")[0]),
            "sentence": sentence_text.text,
            "instance_form_long": " ".join(instance_form_long),
            "instance_form": " ".join([stanza_word.text.lower(), parent.text.lower()]),
            "instance_lemma": " ".join([stanza_word.text.lower(), parent_lemma]),
            "comp1_id": stanza_word.id,
            "comp1_form": stanza_word.text.lower(),
            "comp1_feats": form[0],
            "comp1_lemma": lemma,
            "comp2_id": parent.id,
            "comp2_form": parent.text.lower(),
            "comp2_feats": parent_form[0],
            "comp2_lemma": parent_lemma}
        
        example_sentences.append(dct)
        break


juuni
pühapäev
kolmapäev
maa
linnasüda
keskpäev
lahendus
põhjarannik
august
lõpp
sügis
perekonnaseadus
august
jalanumber
koit
juuni
streetjazz
trend
august
serv
uks
sein
neljapäev
lõpp
rohuneem
laupäev
pühapäev
september
hommik
hõrgutis
kaas
veebruar
reede
lõpp
juuli
reede
sihtpunkt
peainsener


In [48]:
example_sentences[0]

{'sentence_id': 10529383,
 'sentence': 'Eelmisel aastal käis poeg koos isaga Prantsusmaal ja oli muidugi reisist vaimustuses , selles eas poistel on üldse tähtis koos isaga midagi ette võtta .',
 'instance_form_long': 'ja oli muidugi reisist vaimustuses',
 'instance_form': 'reisist vaimustuses',
 'instance_lemma': 'reisist vaimustus',
 'comp1_id': 11,
 'comp1_form': 'reisist',
 'comp1_feats': 'sg el',
 'comp1_lemma': 'reis',
 'comp2_id': 12,
 'comp2_form': 'vaimustuses',
 'comp2_feats': 'sg in',
 'comp2_lemma': 'vaimustus'}

#### II saving result in CSV-format

In [ ]:
with open('../data/results/elative_random_sample_1000.csv', 'w', newline='', encoding="utf-8") as csvfile:
    fieldnames = ['sentence_id', 'sentence', 'instance_form_long', 'instance_form', 'instance_lemma', 'comp1_id', 'comp1_form', 'comp1_feats', 'comp1_lemma', 'comp2_id', 'comp2_form', 'comp2_feats', 'comp2_lemma']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames, delimiter=";")
    writer.writeheader()
    writer.writerows(example_sentences)